In [36]:
# uploaad nessery packages
import os 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


import numpy as np


from scikit_posthocs import posthoc_dunn as dunn
from scipy.stats import mannwhitneyu, false_discovery_control, kruskal, spearmanr

path = os.path.dirname(os.getcwd()).replace('\\', '/') + '/data/'

sns.set_theme(style="whitegrid")



def sinificant_bold_red(val):
    """
    Applies 'font-weight: bold' and 'color: red' if P-value below 0.05.
    """
    if isinstance(val, (int, float)):
        color = "#AA0000" if val <= 0.055 else "#AFAFAF"
        font_weight = 'bold' if val <= 0.055 else 'normal' 
        return f'color: {color}; font-weight: {font_weight}'
        
    return ''

In [ ]:
# upload fluke data
fluke_data = pd.read_excel(path + 'data_four_species.xlsx', 
                           sheet_name='fluke_data').drop('ID', axis=1)
display(fluke_data)

,animal_group,timepoint,liver_flukes,flukes_per_mass,eggs_in_feces,eggs_per_flukes
0,wistar,1_month,0,0.0,0,0.0
1,wistar,1_month,0,0.0,0,0.0
2,wistar,1_month,0,0.0,0,0.0
3,wistar,1_month,0,0.0,0,0.0
4,wistar,1_month,0,0.0,0,0.0
5,balb,1_month,0,0.0,0,0.0
6,balb,1_month,0,0.0,0,0.0
7,balb,1_month,0,0.0,0,0.0
8,balb,1_month,0,0.0,0,0.0
9,balb,1_month,1,0.8,0,0.0


In [ ]:
mwu_data = fluke_data

In [ ]:



wistar, balb, black, hamster = [fluke_data[fluke_data['animal_group'] == i] \
        for i in fluke_data['animal_group'].unique()]

In [77]:
wistar, balb, black, hamster = [fluke_data[fluke_data['animal_group'] == i] \
        for i in fluke_data['animal_group'].unique()]

p_values=[]
for animal in [wistar, balb, black, hamster]:
    
    print(f'1 vs 3 months for animal: \033[1m{animal['animal_group'].unique()[0]}\033[0m')
    
    x=animal[animal['timepoint'] == '1_month']
    y=animal[animal['timepoint'] == '3_month']

    for col in animal.columns[2:]:
        mwu_result = mannwhitneyu(x[col], y[col]).pvalue
        p_values.append(mwu_result)
        
        print(f'For parameter:\033[1m{col}\033[0m Mann-Whitney result: \033[1m{mwu_result:.3f}\033[0m')
      
    print(p_values)   
    p_values = false_discovery_control(p_values, method='bh')
      
    print(p_values)   
    p_values=[]

1 vs 3 months for animal: wistar
For parameter:liver_flukes Mann-Whitney result: 1.000
For parameter:flukes_per_mass Mann-Whitney result: 1.000
For parameter:eggs_in_feces Mann-Whitney result: 1.000
For parameter:eggs_per_flukes Mann-Whitney result: 1.000
[np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0)]
[1. 1. 1. 1.]
1 vs 3 months for animal: balb
For parameter:liver_flukes Mann-Whitney result: 0.071
For parameter:flukes_per_mass Mann-Whitney result: 0.031
For parameter:eggs_in_feces Mann-Whitney result: 0.025
For parameter:eggs_per_flukes Mann-Whitney result: 0.180
[np.float64(0.07064003381435198), np.float64(0.031141210595796758), np.float64(0.025369859822053694), np.float64(0.17971249487899976)]
[0.09418671 0.06228242 0.06228242 0.17971249]
1 vs 3 months for animal: black
For parameter:liver_flukes Mann-Whitney result: 0.010
For parameter:flukes_per_mass Mann-Whitney result: 0.010
For parameter:eggs_in_feces Mann-Whitney result: 0.373
For parameter:eggs_per_fluke

In [46]:
print(wistar)

   animal_group timepoint  liver_flukes  flukes_per_mass  eggs_in_feces  \
0        wistar   1_month             0              0.0              0   
1        wistar   1_month             0              0.0              0   
2        wistar   1_month             0              0.0              0   
3        wistar   1_month             0              0.0              0   
4        wistar   1_month             0              0.0              0   
20       wistar   3_month             0              0.0              0   
21       wistar   3_month             0              0.0              0   
22       wistar   3_month             0              0.0              0   
23       wistar   3_month             0              0.0              0   
24       wistar   3_month             0              0.0              0   

    eggs_per_flukes  
0               0.0  
1               0.0  
2               0.0  
3               0.0  
4               0.0  
20              0.0  
21              0.0 

## Investigation of parasite burden 

### Compare animal groups withing each month

In [43]:
for month in fluke_data['timepoint'].unique():
    
    mask_1 = fluke_data['timepoint'] == month
    temp_data = fluke_data[mask_1]


    wistar, balb, black, hamster = [temp_data[temp_data['animal_group'] == i] \
        for i in temp_data['animal_group'].unique()]

    for col in temp_data.columns[2:]:
        
        print(f'Duration is {month}')
        print(f'Kruscal p-value for {col}')
        print(f'{kruskal(wistar[col], balb[col], 
                        black[col], hamster[col]).pvalue:.4f}\n')
    
        
        dunn_results = dunn(a=temp_data, 
                            group_col='animal_group', 
                            val_col=col,
                            p_adjust='fdr_bh')
        
        styled_dunn = \
            dunn_results.style.map(sinificant_bold_red).format('{:.3f}')
        
        print("Dunn's post-hoc values with fdr-bh:" )
        display(styled_dunn)


Duration is 1_month
Kruscal p-value for liver_flukes
0.0018

Dunn's post-hoc values with fdr-bh:


,balb,black,hamster,wistar
balb,1.000,1.000,0.006,0.765
black,1.000,1.000,0.006,0.765
hamster,0.006,0.006,1.000,0.003
wistar,0.765,0.765,0.003,1.000


Duration is 1_month
Kruscal p-value for flukes_per_mass
0.0018

Dunn's post-hoc values with fdr-bh:


,balb,black,hamster,wistar
balb,1.000,0.950,0.006,0.793
black,0.950,1.000,0.006,0.793
hamster,0.006,0.006,1.000,0.003
wistar,0.793,0.793,0.003,1.000


Duration is 1_month
Kruscal p-value for eggs_in_feces
0.0012

Dunn's post-hoc values with fdr-bh:


...

,balb,black,hamster,wistar
balb,1.000,0.214,0.142,0.150
black,0.214,1.000,0.647,0.008
hamster,0.142,0.647,1.000,0.003
wistar,0.150,0.008,0.003,1.000


Duration is 3_month
Kruscal p-value for eggs_in_feces
0.0015

Dunn's post-hoc values with fdr-bh:


,balb,black,hamster,wistar
balb,1.000,0.338,0.123,0.094
black,0.338,1.000,0.015,0.362
hamster,0.123,0.015,1.000,0.001
wistar,0.094,0.362,0.001,1.000


Duration is 3_month
Kruscal p-value for eggs_per_flukes
0.0030

Dunn's post-hoc values with fdr-bh:


,balb,black,hamster,wistar
balb,1.000,0.815,0.018,0.536
black,0.815,1.000,0.024,0.480
hamster,0.018,0.024,1.000,0.003
wistar,0.536,0.480,0.003,1.000


In [ ]:
liver_data = pd.read_excel(path + 'data_four_species.xlsx', 
                           sheet_name='liver_data').drop('ID', axis=1)
display(liver_data)